<a href="https://colab.research.google.com/github/markjschmidt/PET_List/blob/main/MJS_Claude_Starter_Agent_PET_List_20261109.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/markjschmidt/PET_List/blob/main/MJS_Claude_Starter_Agent_PET_List_20261109.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Starter Claude agent for PET list-mode data

**Prompt 1:** *What is the best way to begin building Agents in Claude? Are there any good Jupyter Notebook examples?*

**Prompt 2:** *Yes, add the starter agent notebook to the repo.*

This notebook builds a small **agent**: Claude gets a set of Python functions ("tools") and decides
for itself which ones to call, in what order, to answer a question about a Philips Vereos listPrint file.

How the agent loop works:

```
 you ask a question
        │
        ▼
 Claude ──► "call load_listprint(path=...)"  ──► your Python runs it ──► result goes back to Claude
        ▲                                                                        │
        └────────────────────────────────────────────────────────────────────────┘
 ... repeats until Claude has enough information, then it writes the final answer
```

The notebook uses the Anthropic Python SDK's **Tool Runner** (`client.beta.messages.tool_runner`), which runs
that loop for you. Section 6 shows the same loop written by hand so you can see what happens underneath.

**Approaches to building Claude agents, from simplest to most capable:**

| Approach | What you write | Use when |
|---|---|---|
| Manual loop (`client.messages.create`) | The whole loop | Learning, or full control |
| **Tool Runner** (this notebook) | Just the tool functions | Most custom-tool agents |
| Managed Agents (beta) | An agent config; Anthropic hosts the loop and a sandbox | Long-running or scheduled agents |
| Claude Agent SDK (`claude-agent-sdk`) | A prompt and options; Claude Code's built-in file, bash and search tools | Coding / file-system agents |

## 1. Install the Anthropic SDK

In [2]:
!pip install -q -U anthropic matplotlib
!pip install -q "pandas==2.2.3"

## 2. Authenticate

1. Create an API key at <https://console.anthropic.com> (API usage is billed separately from a Claude.ai subscription).
2. In Colab, click the **🔑 Secrets** icon in the left sidebar, add a secret named `ANTHROPIC_API_KEY`,
   and switch on **Notebook access**.

Never paste the key into a code cell: notebooks get shared and committed to GitHub.

In [4]:
import os
import anthropic
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-opus-5-5"        # cheaper options: "claude-sonnet-5-5", "claude-haiku-5-5"
print("anthropic SDK version:", anthropic.__version__)

anthropic SDK version: 2.6.0


## 3. Get a listPrint data file

Upload one of the repository's listPrint `.txt` files (for example `3OCLK_0_1000.txt`,
`src_at_bottom_24934.txt` or `listPrint_Randoms2min_250k.txt`).
Each event line looks like:

```
2: PROMPT: xa(55) xb(526) tof(3) za(28) zb(9) lea(138) leb(129)
4: DELAY: xa(55) xb(499) tof(6) za(28) zb(20) lea(138) leb(152)
```

In [5]:
from google.colab import files

uploaded = files.upload()                 # choose a listPrint .txt file
DATA_PATH = "/content/" + next(iter(uploaded))
print("Data file:", DATA_PATH)

# Alternative, if the repository is public: download straight from GitHub instead of uploading
# !wget -q https://raw.githubusercontent.com/markjschmidt/PET_List/main/3OCLK_0_1000.txt -O /content/3OCLK_0_1000.txt
# DATA_PATH = "/content/3OCLK_0_1000.txt"

Saving short_QA_centered.txt to short_QA_centered.txt
Data file: /content/short_QA_centered.txt


## 4. Define the tools

A tool is an ordinary Python function with the `@beta_tool` decorator. The SDK turns it into a tool definition:

* **The function name** becomes the tool name.
* **The type hints** (`path: str`, `tolerance: int = 10`) become the JSON schema for the inputs,
  so Claude knows which arguments to send and what types they must be.
* **The docstring**, including the `Args:` section, is what Claude reads to decide *when* to call the tool
  and *what* to pass. Treat it as instructions to the model and make it precise.

Design rules used below:

* **Tools return strings.** JSON is a good format: compact and unambiguous for Claude to read.
* **Return errors as text instead of raising.** If Claude calls `describe_column` before loading a file,
  it gets back a message telling it what to do and can recover on the next step.
* **Shared state lives outside the tools.** Each call is independent, so the loaded DataFrame goes in the `DATA` dict.
* **Keep results small.** Return summaries, not 250,000 raw rows: everything a tool returns goes into
  Claude's context and is billed as input tokens.

In [6]:
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from anthropic import beta_tool

N_CRYSTALS = 576          # crystals around the Vereos ring
HALF_RING = N_CRYSTALS // 2
TOF_PS_PER_BIN = 19.53    # tofTstampScale, ps per TOF bin

DATA = {}                 # shared state between tool calls

EVENT_RE = re.compile(
    r"(PROMPT|DELAY):\s*xa\((\d+)\) xb\((\d+)\) tof\(([-]?\d+)\) za\((\d+)\) zb\((\d+)\)"
)


def _get_df(event_type: str = "ALL"):
    """Return the loaded DataFrame (optionally filtered), or None if nothing is loaded."""
    df = DATA.get("df")
    if df is None:
        return None
    if event_type.upper() in ("PROMPT", "DELAY"):
        df = df[df["type"] == event_type.upper()]
    return df


def _ring_distance(xa, xb):
    """How far each crystal pair is from an exact 180-degree pair, in crystals (0 = exactly diametric)."""
    d = (np.asarray(xb) - (np.asarray(xa) + HALF_RING)) % N_CRYSTALS
    return np.minimum(d, N_CRYSTALS - d)


@beta_tool
def load_listprint(path: str) -> str:
    """Load a Philips Vereos listPrint .txt file into memory and return a summary.

    Must be called before any other tool. Parses PROMPT and DELAY event lines into a table
    with columns type, xa, xb, tof, za, zb.

    Args:
        path: Path to the listPrint .txt file, e.g. /content/3OCLK_0_1000.txt.
    """
    try:
        rows = []
        with open(path) as f:
            for line in f:
                m = EVENT_RE.search(line)
                if m:
                    rows.append((m.group(1), *map(int, m.groups()[1:])))
    except FileNotFoundError:
        return f"Error: file not found: {path}"
    if not rows:
        return "Error: no PROMPT/DELAY event lines found in the file."
    df = pd.DataFrame(rows, columns=["type", "xa", "xb", "tof", "za", "zb"])
    DATA["df"] = df
    return json.dumps({
        "n_events": len(df),
        "type_counts": df["type"].value_counts().to_dict(),
        "columns": list(df.columns),
    })


@beta_tool
def describe_column(column: str, event_type: str = "ALL") -> str:
    """Return summary statistics (count, mean, std, min, quartiles, max) for one column.

    Args:
        column: One of xa, xb, tof, za, zb. TOF values are in bins (19.53 ps per bin).
        event_type: PROMPT, DELAY, or ALL.
    """
    df = _get_df(event_type)
    if df is None:
        return "Error: no file loaded yet. Call load_listprint first."
    if column not in ("xa", "xb", "tof", "za", "zb"):
        return f"Error: unknown column {column!r}. Use one of xa, xb, tof, za, zb."
    return df[column].describe().round(2).to_json()


@beta_tool
def diametric_fraction(tolerance: int = 10, event_type: str = "PROMPT") -> str:
    """Fraction of events whose crystal pair is within `tolerance` crystals of exactly 180 degrees apart.

    For a source at the center of the ring, xb is close to (xa + 288) % 576 for nearly every true event,
    so a high fraction means a centered source and a low fraction an offset source.

    Args:
        tolerance: Allowed deviation, in crystals, from an exact 180-degree pair (1 crystal = 0.625 degrees).
        event_type: PROMPT, DELAY, or ALL.
    """
    df = _get_df(event_type)
    if df is None:
        return "Error: no file loaded yet. Call load_listprint first."
    d = _ring_distance(df["xa"], df["xb"])
    return json.dumps({
        "event_type": event_type,
        "n_events": len(df),
        "tolerance_crystals": tolerance,
        "fraction_diametric": round(float((d <= tolerance).mean()), 4),
        "median_deviation_crystals": float(np.median(d)),
    })


@beta_tool
def plot_tof_histogram(event_type: str = "ALL") -> str:
    """Draw a histogram of TOF values (in picoseconds) in the notebook, PROMPT and DELAY overlaid,
    and return summary statistics in picoseconds.

    Args:
        event_type: PROMPT, DELAY, or ALL (ALL overlays both types).
    """
    df = _get_df(event_type)
    if df is None:
        return "Error: no file loaded yet. Call load_listprint first."
    fig, ax = plt.subplots(figsize=(8, 4))
    stats = {}
    for etype, color in (("PROMPT", "tab:blue"), ("DELAY", "tab:orange")):
        tof_ps = df.loc[df["type"] == etype, "tof"] * TOF_PS_PER_BIN
        if len(tof_ps) == 0:
            continue
        ax.hist(tof_ps, bins=60, alpha=0.6, color=color, label=f"{etype} (n={len(tof_ps)})")
        stats[etype] = {"mean_ps": round(float(tof_ps.mean()), 1), "std_ps": round(float(tof_ps.std()), 1)}
    ax.set_xlabel("TOF (ps)")
    ax.set_ylabel("Events")
    ax.set_title("TOF distribution")
    ax.legend()
    plt.show()
    return json.dumps({"plotted": True, "stats": stats})


TOOLS = [load_listprint, describe_column, diametric_fraction, plot_tof_histogram]

# See exactly what Claude receives for one tool: name, description and JSON input schema
print(json.dumps(load_listprint.to_dict(), indent=2))

{
  "name": "load_listprint",
  "description": "Load a Philips Vereos listPrint .txt file into memory and return a summary.\n\nMust be called before any other tool. Parses PROMPT and DELAY event lines into a table\nwith columns type, xa, xb, tof, za, zb.",
  "input_schema": {
    "additionalProperties": false,
    "properties": {
      "path": {
        "description": "Path to the listPrint .txt file, e.g. /content/3OCLK_0_1000.txt.",
        "title": "Path",
        "type": "string"
      }
    },
    "required": [
      "path"
    ],
    "type": "object"
  }
}


## 5. Test the tools directly (no API cost)

Before handing tools to Claude, call them yourself. `.call({...})` passes the same kind of input
dictionary that Claude will send. If a tool is broken, this is the cheapest place to find out.

In [7]:
print(load_listprint.call({"path": DATA_PATH}))
print(describe_column.call({"column": "tof", "event_type": "PROMPT"}))
print(diametric_fraction.call({"tolerance": 10}))
print(describe_column.call({"column": "energy"}))   # bad input: returns an error string instead of crashing

{"n_events": 361, "type_counts": {"PROMPT": 357, "DELAY": 4}, "columns": ["type", "xa", "xb", "tof", "za", "zb"]}
{"count":357.0,"mean":0.4,"std":10.44,"min":-45.0,"25%":-6.0,"50%":0.0,"75%":5.0,"max":86.0}
{"event_type": "PROMPT", "n_events": 357, "tolerance_crystals": 10, "fraction_diametric": 0.9636, "median_deviation_crystals": 1.0}
Error: unknown column 'energy'. Use one of xa, xb, tof, za, zb.


## 6. Run the agent

`run_agent()` hands the question and the tools to the **Tool Runner**. Each pass through the `for` loop is one
Claude turn: either a request to call tools (the runner runs them and sends the results back automatically)
or the final answer. Printing each turn shows the agent's reasoning path, i.e. which tools it chose and why.

Parameters worth knowing:

* `system`: standing instructions and domain knowledge. Ring geometry goes here so Claude can interpret
  the numbers the tools return.
* `max_tokens`: upper limit on the length of each Claude reply.
* `max_iterations`: safety limit on the number of loop turns, so a confused agent cannot run (and bill) forever.
* `output_config={"effort": ...}`: how hard Claude thinks (`low`, `medium`, `high`, `xhigh`, `max`).
  This model defaults to `medium`; `high` is a good choice for analysis.
* `fallbacks="default"` (beta): if Claude's safety classifiers decline a request (rare for this kind of work),
  the API automatically retries it on a suitable fallback model instead of returning nothing.

In [ ]:
SYSTEM_PROMPT = """You are a PET physics assistant analyzing Philips Vereos list-mode data.

Scanner geometry: 576 crystals around the ring (IDs 0-575, crystal 0 at 3 o'clock, IDs increasing
clockwise, 0.625 degrees per crystal), 40 crystals axially (za, zb = 0-39), ring diameter 764 mm.
PROMPT events are all coincidences (true + random); DELAY events are randoms from the delayed window.
TOF bins are 19.53 ps; negative TOF means crystal xa received its photon first.

Use the tools to measure things rather than guessing. Report numbers with units, and say briefly
what physical conclusion they support."""


def run_agent(question: str, verbose: bool = True):
    """Run the agent on one question and return the final message."""
    runner = client.beta.messages.tool_runner(
        model=MODEL,
        max_tokens=16000,
        max_iterations=15,
        system=SYSTEM_PROMPT,
        tools=TOOLS,
        output_config={"effort": "high"},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        messages=[{"role": "user", "content": question}],
    )

    final = None
    for turn, message in enumerate(runner, start=1):
        final = message
        for block in message.content:
            if block.type == "text" and block.text.strip():
                print(f"\n🤖 [turn {turn}] {block.text}")
            elif block.type == "tool_use" and verbose:
                print(f"🔧 [turn {turn}] {block.name}({json.dumps(block.input)})")

    if final is not None and final.stop_reason == "refusal":
        print("⚠️ The request was declined:", final.stop_details)
    if final is not None:
        u = final.usage
        print(f"\n(last turn tokens: {u.input_tokens} in / {u.output_tokens} out)")
    return final


answer = run_agent(
    f"Load {DATA_PATH}. Is the source centered in the ring? "
    "Then compare the TOF distributions of PROMPT and DELAY events and explain the difference."
)

### Ask your own questions

Each `run_agent()` call starts a fresh conversation, so it reloads the file itself.
Try questions that need several tools, and look at which tools Claude chooses. Some ideas:

In [ ]:
run_agent(f"Using {DATA_PATH}: what fraction of events are randoms (DELAY)? "
          "Are the axial positions za and zb spread evenly over the 40 crystals?")

## 7. Under the hood: the same agent as a manual loop

The Tool Runner is a convenience. This cell does the same work by hand with the regular
`client.messages.create` call, so you can see the four steps every agent repeats:

1. Send the conversation plus tool definitions to Claude.
2. If `stop_reason == "tool_use"`, run every tool Claude asked for.
3. Append Claude's reply **and** a `tool_result` for each tool call (matched by `tool_use_id`),
   all in one user message.
4. Repeat until Claude stops asking for tools (`stop_reason == "end_turn"`).

The API keeps no memory between calls: the whole `messages` list is sent every time, which is how Claude
"remembers" earlier tool results.

In [ ]:
TOOLS_BY_NAME = {t.name: t for t in TOOLS}


def run_agent_manual(question: str, max_turns: int = 15) -> str:
    messages = [{"role": "user", "content": question}]

    for turn in range(1, max_turns + 1):
        # Step 1: ask Claude, giving it the tool definitions (name, description, input schema)
        response = client.messages.create(
            model=MODEL,
            max_tokens=16000,
            system=SYSTEM_PROMPT,
            tools=[t.to_dict() for t in TOOLS],
            messages=messages,
        )
        # Keep Claude's full reply (text, tool calls and any thinking blocks) in the history
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "refusal":
            return f"Declined: {response.stop_details}"
        if response.stop_reason != "tool_use":
            # Step 4: no more tool calls, so this is the final answer
            return "\n".join(b.text for b in response.content if b.type == "text")

        # Step 2: run each requested tool (Claude may ask for several at once)
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            print(f"🔧 [turn {turn}] {block.name}({json.dumps(block.input)})")
            try:
                result = TOOLS_BY_NAME[block.name].call(block.input)
                is_error = False
            except Exception as exc:              # report failures to Claude instead of crashing
                result, is_error = f"Error: {exc}", True
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,          # must match the id of the tool_use block
                "content": result,
                "is_error": is_error,
            })

        # Step 3: send all results back together in one user message
        messages.append({"role": "user", "content": tool_results})

    return "Stopped: reached max_turns."


print(run_agent_manual(f"Load {DATA_PATH} and tell me whether the source is centered."))

## 8. Next steps

* **Add a tool.** Ideas: a circular ring plot of the LORs (reuse code from the revG/revH notebooks),
  a sinogram, or a function that reads binary `.list` headers (`tofTstampScale`, `c_h_w_b`).
  Write a clear docstring, test it with `.call()`, add it to `TOOLS`.
* **Let Claude write its own analysis code.** Add the server-side code execution tool,
  `{"type": "code_execution_20260521", "name": "code_execution"}`, to `tools` so Claude can run Python in an
  Anthropic-hosted sandbox. For long-running agents with their own workspace, look at **Managed Agents**.
* **Read more:**
  * Anthropic's essay *Building Effective Agents*: <https://www.anthropic.com/engineering/building-effective-agents>
  * Claude Cookbooks (Colab-ready notebooks, see `tool_use/` and `patterns/agents/`): <https://github.com/anthropics/claude-cookbooks>
  * Anthropic Courses, including the tool-use course: <https://github.com/anthropics/courses>
  * Tool use docs: <https://platform.claude.com/docs>
  * Claude Agent SDK: <https://code.claude.com/docs/en/agent-sdk>